# Random Forest Traffic-Sign Image Classification — Sequential Workflow

Dataset layout:

```
dataset/
├── Demo/                     # not used in this notebook
├── TRAIN/
│   ├── 0/                    # folder name = ClassId (no zero-padding)
│   │   ├── 000_1_0001.png    # filename prefix is also zero-padded ClassId
│   │   └── ...
│   ├── 1/
│   └── ...
├── TEST/                     # mixed classes, NO sub-folders
│   ├── 000_1_00.png          # ClassId = zero-padded numeric prefix of filename
│   └── ...
└── labels.csv                # ClassId,Name
```

**Labeling rules:**
- **TRAIN** — the class comes from the **sub-folder name** itself (e.g. folder
  `5` → class `5`). There is no zero-padding at the folder level.
- **TEST** — there are no sub-folders, so the class comes from the
  **zero-padded numeric prefix** at the start of each filename (e.g.
  `000_1_00.png` → class `0`).
- **`labels.csv`** maps `ClassId` → a human-readable `Name` (e.g. `0,Speed
  limit (5km/h)`).
- No ROI / bounding-box columns are available for this dataset, so images
  are used as-is (resized only, no cropping).

Same sequential workflow as before: feature extraction is performed once up
front, then for every feature set (HOG, Color Histogram, Combined) the
notebook runs **Train → Show Training Result → Test → Show Testing Result**
as separate functions/cells, so outputs don't pile up in one place.


In [ ]:
# (Uncomment if running in a fresh environment)
# !pip install opencv-python scikit-image scikit-learn matplotlib seaborn pandas joblib


In [ ]:
import os
import time
import random

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from skimage.feature import hog

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

from IPython.display import display

%matplotlib inline
sns.set_style("whitegrid")


## 1. Configuration

In [ ]:
# --- Dataset locations ----------------------------------------------------
DATASET_DIR = "dataset"
TRAIN_DIR = os.path.join(DATASET_DIR, "TRAIN")   # sub-folders named by ClassId, e.g. "0", "1", ...
TEST_DIR = os.path.join(DATASET_DIR, "TEST")     # mixed images, no sub-folders
LABELS_CSV = os.path.join(DATASET_DIR, "labels.csv")   # ClassId,Name
# DATASET_DIR also contains a "Demo" folder which is not used in this notebook.

MODEL_DIR = "saved_models"             # trained models (.pkl) are stored here
FEATURE_CACHE_DIR = os.path.join(MODEL_DIR, "feature_cache")
FORCE_RECOMPUTE_FEATURES = False       # set True to ignore any cached features

# --- Feature extraction settings ------------------------------------------
IMG_SIZE = (64, 64)        # every image is resized to this size before extraction
COLOR_BINS = (8, 8, 8)     # H, S, V histogram bins for the color-histogram feature

# --- Optional speed-up while iterating -------------------------------------
SUBSAMPLE_TRAIN_PER_CLASS = None    # e.g. 300
SUBSAMPLE_TEST_TOTAL = None         # e.g. 3000

# --- Reproducibility ---------------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FEATURE_CACHE_DIR, exist_ok=True)


## 2. Class Names (from `labels.csv`)

In [ ]:
labels_df = pd.read_csv(LABELS_CSV)
class_id_to_name = dict(zip(labels_df['ClassId'].astype(int), labels_df['Name']))

print(f"Loaded {len(class_id_to_name)} class names, e.g.:")
for cid in sorted(class_id_to_name)[:5]:
    print(f"  {cid}: {class_id_to_name[cid]}")


## 3. Loading Image Lists & Labels

- **TRAIN**: the label is simply the sub-folder name (`int(folder_name)`).
- **TEST**: the label is parsed from the **zero-padded numeric prefix**
  before the first underscore in the filename (e.g. `"000_1_00.png"` →
  `int("000")` → class `0`).


In [ ]:
def load_train_dataset(train_dir):
    """TRAIN/<ClassId>/*.png -> (paths, labels), label = folder name."""
    valid_ext = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
    paths, labels = [], []
    for entry in sorted(os.listdir(train_dir)):
        full_path = os.path.join(train_dir, entry)
        if not os.path.isdir(full_path) or not entry.isdigit():
            continue
        class_id = int(entry)
        for fname in sorted(os.listdir(full_path)):
            if fname.lower().endswith(valid_ext):
                paths.append(os.path.join(full_path, fname))
                labels.append(class_id)
    return paths, labels


def parse_class_id_from_filename(fname):
    """'000_1_00.png' -> 0 (zero-padded numeric prefix before the first '_')."""
    prefix = fname.split('_')[0]
    return int(prefix)


def load_test_dataset(test_dir):
    """TEST/*.png (mixed) -> (paths, labels), label parsed from filename prefix."""
    valid_ext = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
    paths, labels = [], []
    for fname in sorted(os.listdir(test_dir)):
        if not fname.lower().endswith(valid_ext):
            continue
        full_path = os.path.join(test_dir, fname)
        try:
            class_id = parse_class_id_from_filename(fname)
        except ValueError:
            print(f"  [warn] Could not parse a class id from filename '{fname}', skipping.")
            continue
        paths.append(full_path)
        labels.append(class_id)
    return paths, labels


train_paths, train_labels = load_train_dataset(TRAIN_DIR)
test_paths, test_labels = load_test_dataset(TEST_DIR)

print(f"Training images resolved: {len(train_paths)}  |  classes: {len(set(train_labels))}")
print(f"Testing images resolved : {len(test_paths)}  |  classes: {len(set(test_labels))}")


In [ ]:
def subsample_per_class(paths, labels, cap, seed=RANDOM_STATE):
    if cap is None:
        return paths, labels
    rng = np.random.RandomState(seed)
    by_class = {}
    for p, l in zip(paths, labels):
        by_class.setdefault(l, []).append(p)
    new_paths, new_labels = [], []
    for l, items in by_class.items():
        chosen = rng.permutation(len(items))[:cap]
        for i in chosen:
            new_paths.append(items[i]); new_labels.append(l)
    return new_paths, new_labels


def subsample_total(paths, labels, cap, seed=RANDOM_STATE):
    if cap is None or cap >= len(paths):
        return paths, labels
    rng = np.random.RandomState(seed)
    chosen = rng.choice(len(paths), size=cap, replace=False)
    return [paths[i] for i in chosen], [labels[i] for i in chosen]


train_paths, train_labels = subsample_per_class(train_paths, train_labels, SUBSAMPLE_TRAIN_PER_CLASS)
test_paths, test_labels = subsample_total(test_paths, test_labels, SUBSAMPLE_TEST_TOTAL)

print(f"Using {len(train_paths)} training images and {len(test_paths)} testing images "
      f"after any subsampling.")


In [ ]:
# Quick sanity peek: a handful of random training images with their labels
sample_idx = np.random.choice(len(train_paths), size=min(5, len(train_paths)), replace=False)
fig, axes = plt.subplots(1, len(sample_idx), figsize=(3 * len(sample_idx), 3))
axes = np.atleast_1d(axes)
for ax, i in zip(axes, sample_idx):
    img = cv2.cvtColor(cv2.imread(train_paths[i]), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    cid = train_labels[i]
    ax.set_title(f"{cid}: {class_id_to_name.get(cid, '?')}", fontsize=9)
    ax.axis('off')
plt.suptitle("Sample training images")
plt.tight_layout()
plt.show()


In [ ]:
def show_class_reference_grid(paths, labels, class_id_to_name, max_classes=43):
    """Shows one example image per class (first occurrence found in `paths`)."""
    first_path_per_class = {}
    for p, l in zip(paths, labels):
        if l not in first_path_per_class:
            first_path_per_class[l] = p

    class_ids = sorted(first_path_per_class)[:max_classes]
    n_cols = 10
    n_rows = int(np.ceil(len(class_ids) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.6, n_rows * 1.9))
    axes = np.atleast_1d(axes).reshape(-1)

    for ax, cid in zip(axes, class_ids):
        img = cv2.cvtColor(cv2.imread(first_path_per_class[cid]), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"{cid}: {class_id_to_name.get(cid, '?')}", fontsize=6)
        ax.axis('off')
    for ax in axes[len(class_ids):]:
        ax.axis('off')

    fig.suptitle("Class reference (one example per class from TRAIN)", fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


show_class_reference_grid(train_paths, train_labels, class_id_to_name)


## 4. Feature Extraction (performed once, up front, for all feature sets)

Two extraction methods:

- **HOG** — local shape/edge/gradient structure (grayscale)
- **Color Histogram (HSV)** — color distribution

Both are computed together per image (resize only, no cropping — this
dataset has no ROI/bounding-box info) so they always refer to exactly the
same set of successfully-read images. Results are cached to disk with
`joblib` so this doesn't need to rerun every time.


In [ ]:
def extract_hog(gray_resized_img):
    return hog(
        gray_resized_img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        feature_vector=True,
    )


def extract_color_histogram(color_resized_img, bins=COLOR_BINS):
    hsv = cv2.cvtColor(color_resized_img, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, [0, 180, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()


def build_dual_features(paths, labels, img_size=IMG_SIZE,
                         cache_path=None, force_recompute=False):
    if cache_path and os.path.isfile(cache_path) and not force_recompute:
        print(f"  Loading cached features from {cache_path}")
        return joblib.load(cache_path)

    hog_feats, color_feats, valid_labels, valid_paths = [], [], [], []
    for path, label in zip(paths, labels):
        img = cv2.imread(path)
        if img is None:
            print(f"  [skip] could not read: {path}")
            continue
        img_r = cv2.resize(img, img_size)
        gray = cv2.cvtColor(img_r, cv2.COLOR_BGR2GRAY)

        hog_feats.append(extract_hog(gray))
        color_feats.append(extract_color_histogram(img_r))
        valid_labels.append(label)
        valid_paths.append(path)

    result = (np.array(hog_feats), np.array(color_feats), np.array(valid_labels), valid_paths)
    if cache_path:
        joblib.dump(result, cache_path)
        print(f"  Cached features to {cache_path}")
    return result


In [ ]:
print("Extracting/loading features for TRAINING images...")
X_train_hog, X_train_color, y_train, train_paths_valid = build_dual_features(
    train_paths, train_labels,
    cache_path=os.path.join(FEATURE_CACHE_DIR, "train_features.joblib"),
    force_recompute=FORCE_RECOMPUTE_FEATURES,
)

print("Extracting/loading features for TESTING images...")
X_test_hog, X_test_color, y_test, test_paths_valid = build_dual_features(
    test_paths, test_labels,
    cache_path=os.path.join(FEATURE_CACHE_DIR, "test_features.joblib"),
    force_recompute=FORCE_RECOMPUTE_FEATURES,
)

# Combined feature set: concatenate HOG + Color Histogram
X_train_combined = np.hstack([X_train_hog, X_train_color])
X_test_combined = np.hstack([X_test_hog, X_test_color])

# All classes present across train + test, used for consistent report/plot ordering
ALL_LABELS = sorted(set(np.unique(y_train)) | set(np.unique(y_test)))

print(f"HOG feature length     : {X_train_hog.shape[1]}")
print(f"Color feature length   : {X_train_color.shape[1]}")
print(f"Combined feature length: {X_train_combined.shape[1]}")


## 5. Model Declaration

Declares the Random Forest hyperparameter search space and cross-validation
settings shared by every feature set. No training happens in this cell.


In [ ]:
# Hyperparameter grid searched for every feature set.
PARAM_GRID = {
    'n_estimators': [100, 200],
    'max_depth': [None, 15, 30],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

N_SPLITS = 5                # K for K-Fold cross-validation
VAL_HOLDOUT_FRACTION = 0.2  # fraction of training images reserved for validation


def create_base_model(random_state=RANDOM_STATE):
    """Factory for a fresh, untrained Random Forest — used inside grid search."""
    return RandomForestClassifier(random_state=random_state)


## 6. Training Functions

Small, single-purpose functions instead of one large function:

- `split_train_validation` — holds out validation images from training
- `run_grid_search` — Stratified K-Fold + hyperparameter grid search
- `evaluate_on_validation` — predicts & scores on the held-out validation set
- `train_final_model` — retrains with the best hyperparameters on ALL training data
- `save_model` — persists the final model to disk (feature-specific filename)
- `train_on_feature` — orchestrates the steps above for one feature set


In [ ]:
def split_train_validation(X, y, image_paths, val_fraction=VAL_HOLDOUT_FRACTION,
                            random_state=RANDOM_STATE):
    X_tr, X_val, y_tr, y_val, paths_tr, paths_val = train_test_split(
        X, y, image_paths, test_size=val_fraction, stratify=y, random_state=random_state
    )
    return X_tr, X_val, y_tr, y_val, paths_tr, paths_val


In [ ]:
def run_grid_search(X_tr, y_tr, param_grid=PARAM_GRID, n_splits=N_SPLITS,
                     random_state=RANDOM_STATE):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    grid_search = GridSearchCV(
        estimator=create_base_model(random_state),
        param_grid=param_grid,
        cv=cv,
        scoring='f1_weighted',
        n_jobs=-1,
        return_train_score=True,
    )
    grid_search.fit(X_tr, y_tr)
    return grid_search


In [ ]:
def evaluate_on_validation(model, X_val, y_val):
    val_preds = model.predict(X_val)
    val_acc = accuracy_score(y_val, val_preds)
    val_f1 = f1_score(y_val, val_preds, average='weighted')
    return val_preds, val_acc, val_f1


In [ ]:
def train_final_model(best_params, X_train_full, y_train_full, random_state=RANDOM_STATE):
    model = RandomForestClassifier(**best_params, random_state=random_state)
    model.fit(X_train_full, y_train_full)
    return model


def save_model(model, feature_name, model_dir=MODEL_DIR):
    model_path = os.path.join(model_dir, f"rf_model_{feature_name}.pkl")
    joblib.dump(model, model_path)
    return model_path


In [ ]:
def train_on_feature(feature_name, X_train_full, y_train_full, train_image_paths,
                      param_grid=PARAM_GRID, n_splits=N_SPLITS,
                      val_fraction=VAL_HOLDOUT_FRACTION, random_state=RANDOM_STATE):
    """Orchestrates: val split -> grid search -> validate -> retrain on full data -> save."""
    print(f"[Train] feature set = {feature_name}")

    X_tr, X_val, y_tr, y_val, _, _ = split_train_validation(
        X_train_full, y_train_full, train_image_paths, val_fraction, random_state
    )

    grid_search = run_grid_search(X_tr, y_tr, param_grid, n_splits, random_state)
    val_preds, val_acc, val_f1 = evaluate_on_validation(grid_search.best_estimator_, X_val, y_val)

    final_model = train_final_model(grid_search.best_params_, X_train_full, y_train_full, random_state)
    model_path = save_model(final_model, feature_name)

    print(f"[Train] best CV F1 = {grid_search.best_score_:.4f} | validation F1 = {val_f1:.4f}")
    print(f"[Train] final model saved to: {model_path}")

    return {
        'feature_name': feature_name,
        'grid_search': grid_search,
        'model': final_model,
        'model_path': model_path,
        'X_val': X_val,
        'y_val': y_val,
        'val_preds': val_preds,
        'val_accuracy': val_acc,
        'val_f1': val_f1,
    }


## 7. Testing Functions

- `compute_classification_metrics` — accuracy/F1/precision/recall
- `test_on_feature` — predicts on the test set, times inference, scores it


In [ ]:
def compute_classification_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, average='weighted'),
        'precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
    }


In [ ]:
def test_on_feature(model, X_test, y_test, feature_name):
    print(f"[Test] feature set = {feature_name}")

    start = time.time()
    test_preds = model.predict(X_test)
    elapsed = time.time() - start
    avg_ms = (elapsed / len(X_test)) * 1000
    throughput = len(X_test) / elapsed if elapsed > 0 else float('inf')

    metrics = compute_classification_metrics(y_test, test_preds)

    print(f"[Test] accuracy = {metrics['accuracy']:.4f} | f1 = {metrics['f1']:.4f} | "
          f"{avg_ms:.4f} ms/image ({throughput:.2f} img/sec)")

    return {
        'feature_name': feature_name,
        'y_test': y_test,
        'test_preds': test_preds,
        'metrics': metrics,
        'total_inference_time_s': elapsed,
        'avg_inference_time_ms': avg_ms,
        'throughput_img_per_sec': throughput,
    }


## 8. Display Functions (pure display — no training/testing logic)

- `display_image_grid` — low-level: renders one grid of images with T/P labels
- `show_training_result` — prints/plots grid-search + validation results
- `show_testing_result` — prints/plots test metrics + confusion matrix
- `show_prediction_gallery` — up to 100 random success cases + all failure cases


In [ ]:
def display_image_grid(indices, image_paths, y_true, y_pred, class_id_to_name,
                        title, color='green', max_cols=10, cell_size=2.1):
    """Displays a grid of images with True(T)/Predicted(P) class names."""
    n = len(indices)
    if n == 0:
        print(f"[{title}] Nothing to display.")
        return

    n_cols = min(max_cols, n)
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * cell_size, n_rows * (cell_size + 0.5)))
    axes = np.atleast_1d(axes).reshape(-1)

    for ax_i, idx in enumerate(indices):
        img = cv2.cvtColor(cv2.imread(image_paths[idx]), cv2.COLOR_BGR2RGB)
        axes[ax_i].imshow(img)
        true_id, pred_id = int(y_true[idx]), int(y_pred[idx])
        true_name = class_id_to_name.get(true_id, f"Class {true_id}")
        pred_name = class_id_to_name.get(pred_id, f"Class {pred_id}")
        axes[ax_i].set_title(f"T:{true_name}\nP:{pred_name}", fontsize=7, color=color)
        axes[ax_i].axis('off')

    for ax_i in range(n, len(axes)):
        axes[ax_i].axis('off')

    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [ ]:
def show_training_result(train_output, class_id_to_name, all_labels):
    feature_name = train_output['feature_name']
    grid_search = train_output['grid_search']

    print("=" * 90)
    print(f"TRAINING RESULT — {feature_name.upper()}")
    print("=" * 90)
    print(f"Best hyperparameters : {grid_search.best_params_}")
    print(f"Best CV F1 (weighted): {grid_search.best_score_:.4f}")

    cv_results = pd.DataFrame(grid_search.cv_results_)
    cv_results_sorted = cv_results.sort_values('mean_test_score', ascending=False)
    print("\nTop 10 hyperparameter combinations:")
    display(cv_results_sorted[['params', 'mean_test_score', 'std_test_score', 'mean_train_score']].head(10))

    top10 = cv_results_sorted.head(10).iloc[::-1]
    plt.figure(figsize=(10, 5))
    plt.barh(range(len(top10)), top10['mean_test_score'], xerr=top10['std_test_score'], color='steelblue')
    plt.yticks(range(len(top10)), [str(p) for p in top10['params']], fontsize=7)
    plt.xlabel('Mean CV F1-weighted score')
    plt.title(f'Top 10 Hyperparameter Combinations — {feature_name}')
    plt.tight_layout()
    plt.show()

    y_val, val_preds = train_output['y_val'], train_output['val_preds']
    target_names = [class_id_to_name.get(c, f"Class {c}") for c in all_labels]

    print(f"\nValidation — Accuracy: {train_output['val_accuracy']:.4f} | "
          f"F1 (weighted): {train_output['val_f1']:.4f}")
    print(classification_report(y_val, val_preds, labels=all_labels,
                                 target_names=target_names, zero_division=0))

    cm_val = confusion_matrix(y_val, val_preds, labels=all_labels)
    side = min(0.35 * len(all_labels) + 3, 16)
    plt.figure(figsize=(side, side))
    sns.heatmap(cm_val, cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    plt.title(f'Validation Confusion Matrix — {feature_name}')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.xticks(fontsize=6, rotation=90); plt.yticks(fontsize=6)
    plt.tight_layout()
    plt.show()


In [ ]:
def show_testing_result(test_output, class_id_to_name, all_labels):
    feature_name = test_output['feature_name']
    y_test, test_preds = test_output['y_test'], test_output['test_preds']
    metrics = test_output['metrics']
    target_names = [class_id_to_name.get(c, f"Class {c}") for c in all_labels]

    print("=" * 90)
    print(f"TESTING RESULT — {feature_name.upper()}")
    print("=" * 90)
    print(f"Accuracy          : {metrics['accuracy']:.4f}")
    print(f"F1 (weighted)     : {metrics['f1']:.4f}")
    print(f"Precision         : {metrics['precision']:.4f}")
    print(f"Recall            : {metrics['recall']:.4f}")
    print(f"Total inference   : {test_output['total_inference_time_s']:.4f}s for {len(y_test)} images")
    print(f"Avg inference time: {test_output['avg_inference_time_ms']:.4f} ms/image | "
          f"Throughput: {test_output['throughput_img_per_sec']:.2f} images/sec")
    print("\nClassification report:\n",
          classification_report(y_test, test_preds, labels=all_labels,
                                 target_names=target_names, zero_division=0))

    cm_test = confusion_matrix(y_test, test_preds, labels=all_labels)
    side = min(0.35 * len(all_labels) + 3, 16)
    plt.figure(figsize=(side, side))
    sns.heatmap(cm_test, cmap='Greens', xticklabels=target_names, yticklabels=target_names)
    plt.title(f'Test Confusion Matrix — {feature_name}')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.xticks(fontsize=6, rotation=90); plt.yticks(fontsize=6)
    plt.tight_layout()
    plt.show()


In [ ]:
def show_prediction_gallery(test_output, test_image_paths, class_id_to_name):
    feature_name = test_output['feature_name']
    y_test, test_preds = test_output['y_test'], test_output['test_preds']

    correct_idx = np.where(test_preds == y_test)[0]
    n_show = min(100, len(correct_idx))
    chosen = np.random.choice(correct_idx, size=n_show, replace=False) if n_show > 0 else np.array([], dtype=int)
    display_image_grid(
        chosen, test_image_paths, y_test, test_preds, class_id_to_name,
        title=f"Random Success Cases — {feature_name} ({n_show} of {len(correct_idx)} shown)",
        color='green'
    )

    wrong_idx = np.where(test_preds != y_test)[0]
    print(f"\nTotal failed cases: {len(wrong_idx)} / {len(y_test)}")
    display_image_grid(
        wrong_idx, test_image_paths, y_test, test_preds, class_id_to_name,
        title=f"All Failure Cases — {feature_name} ({len(wrong_idx)} shown)",
        color='red'
    )


## 9. HOG — Train

In [ ]:
train_output_hog = train_on_feature(
    feature_name="hog",
    X_train_full=X_train_hog, y_train_full=y_train,
    train_image_paths=train_paths_valid,
)


## 10. HOG — Show Training Result

In [ ]:
show_training_result(train_output_hog, class_id_to_name, ALL_LABELS)

## 11. HOG — Test

In [ ]:
test_output_hog = test_on_feature(
    model=train_output_hog['model'],
    X_test=X_test_hog, y_test=y_test,
    feature_name="hog",
)


## 12. HOG — Show Testing Result

In [ ]:
show_testing_result(test_output_hog, class_id_to_name, ALL_LABELS)

In [ ]:
show_prediction_gallery(test_output_hog, test_paths_valid, class_id_to_name)

## 13. Color Histogram — Train

In [ ]:
train_output_color = train_on_feature(
    feature_name="color_histogram",
    X_train_full=X_train_color, y_train_full=y_train,
    train_image_paths=train_paths_valid,
)


## 14. Color Histogram — Show Training Result

In [ ]:
show_training_result(train_output_color, class_id_to_name, ALL_LABELS)

## 15. Color Histogram — Test

In [ ]:
test_output_color = test_on_feature(
    model=train_output_color['model'],
    X_test=X_test_color, y_test=y_test,
    feature_name="color_histogram",
)


## 16. Color Histogram — Show Testing Result

In [ ]:
show_testing_result(test_output_color, class_id_to_name, ALL_LABELS)

In [ ]:
show_prediction_gallery(test_output_color, test_paths_valid, class_id_to_name)

## 17. Combined (HOG + Color Histogram) — Train

In [ ]:
train_output_combined = train_on_feature(
    feature_name="combined_hog_color",
    X_train_full=X_train_combined, y_train_full=y_train,
    train_image_paths=train_paths_valid,
)


## 18. Combined — Show Training Result

In [ ]:
show_training_result(train_output_combined, class_id_to_name, ALL_LABELS)

## 19. Combined — Test

In [ ]:
test_output_combined = test_on_feature(
    model=train_output_combined['model'],
    X_test=X_test_combined, y_test=y_test,
    feature_name="combined_hog_color",
)


## 20. Combined — Show Testing Result

In [ ]:
show_testing_result(test_output_combined, class_id_to_name, ALL_LABELS)

In [ ]:
show_prediction_gallery(test_output_combined, test_paths_valid, class_id_to_name)

## 21. Compare All Three Models

Each model was trained and saved separately under `saved_models/`:

- `rf_model_hog.pkl`
- `rf_model_color_histogram.pkl`
- `rf_model_combined_hog_color.pkl`


In [ ]:
summary_rows = []
for train_out, test_out in [
    (train_output_hog, test_output_hog),
    (train_output_color, test_output_color),
    (train_output_combined, test_output_combined),
]:
    summary_rows.append({
        'feature': train_out['feature_name'],
        'best_params': train_out['grid_search'].best_params_,
        'cv_best_f1': train_out['grid_search'].best_score_,
        'val_accuracy': train_out['val_accuracy'],
        'val_f1': train_out['val_f1'],
        'test_accuracy': test_out['metrics']['accuracy'],
        'test_f1': test_out['metrics']['f1'],
        'test_precision': test_out['metrics']['precision'],
        'test_recall': test_out['metrics']['recall'],
        'avg_inference_time_ms': test_out['avg_inference_time_ms'],
        'throughput_img_per_sec': test_out['throughput_img_per_sec'],
        'model_path': train_out['model_path'],
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
plot_specs = [
    ('test_accuracy', 'Test Accuracy'),
    ('test_f1', 'Test F1 (weighted)'),
    ('throughput_img_per_sec', 'Throughput (images/sec)'),
]
colors = ['#4C72B0', '#DD8452', '#55A868']
for ax, (metric, title) in zip(axes, plot_specs):
    ax.bar(summary_df['feature'], summary_df[metric], color=colors)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

print("\nAll trained models are stored separately under:", os.path.abspath(MODEL_DIR))
